<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/Phase1_FixedDistanceGrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Fixed distances for HRP

**Research question:** Which fixed distance measure produces the strongest out-of-sample performance within the Hierarchical Risk Parity framework?

The experiment uses ten stocks, 252-observation training windows and 63-observation test windows. The initial training date is in 2005; test returns start after the first training window.

**Reading the results:** 79 test windows, 21 August 2006–3 June 2026. Pooled metrics and paired window differences answer different questions. Run top-to-bottom in a fresh runtime; historical adjusted prices and unpinned libraries may change on a later rerun.


## Notebook guide

1. Dependencies and universe
2. Data and walk-forward windows
3. Shared HRP allocator
4. Performance metrics and paired comparisons
5. Five distance candidates
6. Cross-candidate summaries and Holm correction
7. PCA component sensitivity
8. Chronological half-split
9. Market regime comparisons
10. Summary chart


## 1. Dependencies and universe

Install the data and modeling libraries, then define a consistent ticker order. The global NumPy seed does not control every estimator or random generator: bootstrap helpers use their own explicit seed.


In [ ]:
# Install and import the shared data, clustering and distance-model libraries.

!pip install yfinance scikit-learn --quiet
!pip install lingam --only-binary=:all: -q
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from lingam import VARLiNGAM
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')
np.random.seed(0)

In [ ]:
# Define the manually selected stock universe. Dictionary order determines SYMBOLS,
# the common row/column order for distances, covariance and portfolio weights.

TICKERS = {
    'Energy': 'XOM', 'Materials': 'LIN', 'Industrials': 'HON',
    'Consumer Discretionary': 'AMZN', 'Consumer Staples': 'PG',
    'Healthcare': 'JNJ', 'Financials': 'JPM',
    'Information Technology': 'MSFT', 'Communication Services': 'GOOGL',
    'Utilities': 'NEE',
}
SYMBOLS = list(TICKERS.values())
N = len(SYMBOLS)
print(f"Universe: {SYMBOLS}")

## 2. Data and walk-forward windows

*   Download adjusted daily closing prices for the manually chosen ten-stock universe.
*   Forward-fill missing prices, calculate simple returns and retain dates complete across all assets.
* An internal filled price produces a zero daily return; leading gaps remain unavailable.
* Use 252 observations for training and the following 63 for testing.
* Advance by 63 observations each time: training windows overlap, while test windows do not.
* Disjoint test periods do not guarantee statistical independence.

* Specifying dates fixes the requested interval, but historical adjusted prices may still change between downloads.


In [ ]:
# Download prices and build chronological, observation-count-based folds.
# The requested end date is exclusive; the first printed date is a training date.

#downloading adjusted closing prices
raw = yf.download(SYMBOLS, start='2004-01-01', end='2026-06-10', auto_adjust=True)['Close']
raw = raw[SYMBOLS]
# fill missing prices using most recent available price
# Carry only earlier observed prices forward. Internal missing-price days then
# produce zero simple returns; leading missing history remains unavailable.
prices = raw.ffill()
#calculate daily simple returns and retain dates with returns for all assets
returns = prices.pct_change().dropna()
print(f"Returns shape: {returns.shape}")

#each window uses roughly one trading yr to train and one quarter to test
TRAIN_DAYS = 252
TEST_DAYS = 63
FIRST_TRAIN_START = '2005-08-19'


def build_windows(returns, train_days, test_days, first_train_start):
    # Return rolling training and test date boundaries.

    idx = returns.index
    #if requested date is absent, use next avail return date
    start = idx.searchsorted(pd.Timestamp(first_train_start))
    windows, pos = [], start

    #only include windows with complete training and test periods
    while pos + train_days + test_days <= len(idx):
      #store inclusive boundaries (train start, train end)
      #test start, test end. testing begins after training ends.
        windows.append((idx[pos], idx[pos+train_days-1],
                        idx[pos+train_days], idx[pos+train_days+test_days-1]))

        #move fwd by one test period
        #trg windows overlap, but test windows don't

        pos += test_days

    return windows

windows = build_windows(returns, TRAIN_DAYS, TEST_DAYS, FIRST_TRAIN_START)


#check number of windows and overall date coverage
#each printed range runs from trg start to test end

print(f"Total windows: {len(windows)}")
print(f"First: {windows[0][0].date()} - {windows[0][3].date()}")
print(f"Last:  {windows[-1][0].date()} - {windows[-1][3].date()}")

## 3. Shared HRP allocator

* Each candidate supplies a symmetric asset-by-asset dissimilarity matrix.
* Single-linkage clustering produces a leaf order, which recursive bisection repeatedly splits at its midpoint.
* The side with lower estimated cluster variance receives more weight.

* Cluster variance uses inverse-variance weights and the **raw training-return covariance**, regardless of how the distance was constructed.
* Bisection follows halves of the ordered list, not necessarily the dendrogram’s child groups.

* The distance and covariance are aligned to `SYMBOLS`.
* Single linkage can accept dissimilarities without a triangle-inequality guarantee; the causal-strength transformations below should not automatically be described as mathematical metrics.


In [ ]:
# Common allocation procedure: distance -> single linkage -> leaf order -> bisection.
# Only the asset order depends on distance; allocation risk uses raw covariance.

def quasi_diag(link, num_items):
    # Leaf IDs are below num_items; larger IDs refer to merged clusters.
    # Only child identifiers are used after this cast, not the truncated merge distances.
    link = link.astype(int)
    sort_ix = pd.Series([link[-1, 0], link[-1, 1]])
    #start from final merge (root)
    while sort_ix.max() >= num_items:
        # Space positions apart so the two children can replace a merged-cluster entry
        # while preserving left-to-right traversal order.
        sort_ix.index = range(0, sort_ix.shape[0] * 2, 2)
        #entries pointing at merged cluster, not leaf
        df0 = sort_ix[sort_ix >= num_items]
        #look up cluster's two children in linkage matrix
        i = df0.index
        j = df0.values - num_items
        #replace with first child
        sort_ix[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        #insert second child after
        sort_ix = pd.concat([sort_ix, df0]).sort_index()
        sort_ix.index = range(sort_ix.shape[0])
    #fully expanded leaf ordering
    return sort_ix.tolist()

def cluster_var(cov_slice):
    #inverse-variance weights within this cluster
    # Use inverse individual variances as within-cluster weights, then calculate
    # their portfolio variance using the full cluster covariance. Positive variances are assumed.
    ivp = 1.0 / np.diag(cov_slice.values)
    ivp /= ivp.sum()
    #cluster's own variance under those weights
    return ivp @ cov_slice.values @ ivp

def get_rec_bipart(cov, sort_ix):
    w = pd.Series(1.0, index=sort_ix)
    c_items = [sort_ix]
    while len(c_items) > 0:
      #splitting every current cluster in half, following quasi-diagonal ordering
        c_items = [i[j:k] for i in c_items for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) if len(i) > 1]
        for i in range(0, len(c_items), 2):
            c0, c1 = c_items[i], c_items[i + 1]
            w0 = cluster_var(cov.loc[c0, c0])
            w1 = cluster_var(cov.loc[c1, c1])
            #lower-variance side gets the larger share
            # Left share = right cluster variance / sum of both cluster variances.
            # Each split multiplies existing asset weights; cross-half covariance is omitted.
            alpha = 1 - w0 / (w0 + w1)
            w[c0] *= alpha
            w[c1] *= 1 - alpha
    return w

def hrp_weights_from_dist(returns_df, dist_df):
    # Ensure symbol alignment and remove any potential NaNs in training
    clean_rets = returns_df[SYMBOLS].dropna(axis=0)
    cov = clean_rets.cov()
    # Ensure distance matrix is aligned with the symbols
    dist_vals = dist_df.loc[SYMBOLS, SYMBOLS].values
    # Extract the upper-triangle vector expected by linkage. checks=False skips
    # symmetry and zero-diagonal checks, so valid finite input is assumed.
    condensed = squareform(dist_vals, checks=False)
    link = linkage(condensed, method='single')
    sort_ix_pos = quasi_diag(link, len(SYMBOLS))
    sort_ix = [SYMBOLS[i] for i in sort_ix_pos]
    w = get_rec_bipart(cov, sort_ix)
    return (w / w.sum()).reindex(SYMBOLS)

def get_corr_distance(returns_df):
    corr = returns_df[SYMBOLS].corr()
    return np.sqrt((1 - corr) / 2)

print("HRP harness refined for index alignment.")

## 4. Performance metrics and paired comparisons

- **Sharpe:** annualized mean divided by volatility with a five-lag Bartlett-weighted serial-correlation adjustment. The risk-free rate is implicitly zero.
- **Maximum drawdown:** the most negative decline from a recorded wealth peak; closer to zero is better.
- **CEQ:** annualized arithmetic mean minus `gamma / 2` times annualized variance. Higher gamma penalizes risk more heavily.
- **Paired bootstrap:** resample four-window circular blocks of candidate-minus-baseline scores using 5,000 draws and seed 42. Report a 95% basic interval and a null-centred two-sided test with a finite-simulation correction. This matches main Phase 2 and the second universe. Local dependence is preserved within blocks; longer-range dependence and model-selection effects are not fully addressed.


In [ ]:
# Shared daily-return metrics and circular block resampling of paired window scores.
# The bootstrap checks matching dates and finite metric differences.

# Evaluation protocol: metrics and significance test
def sharpe_lo2002(returns_series, periods=252, n_lags=5):
    #calculate mean and sample variance of daily returns
    mu, var = returns_series.mean(), returns_series.var()

    #estimate serial dependence using first five lags by default
    #bartlett weights give less weight to more distant lags

    adj = 0.0
    for k in range(1, n_lags + 1):
        rho = returns_series.autocorr(lag=k)
        if not np.isnan(rho):
            adj += (1 - k / (n_lags + 1)) * rho

    #adjust variance for serial dependence and annualise the ratio
    #the floor prevents taking the sq root of zero or negative value

    # The denominator floor avoids invalid square roots but may produce extreme
    # ratios when the estimated adjusted variance is near zero or negative.
    # No risk-free return is subtracted from the mean.
    return (mu / np.sqrt(max(var * (1 + 2 * adj), 1e-12))) * np.sqrt(periods)


def max_drawdown(r):
    # Corrected 2026-09-09: initial wealth of 1 is now prepended to the wealth
    # path, so a loss on the very first observation is no longer excluded from
    # the running peak used to compute drawdown.
    cum = pd.concat([pd.Series([1.0]), (1 + r).cumprod()], ignore_index=True)
    peak = cum.cummax()
    return ((cum - peak) / peak).min()

def ceq(r, gamma, periods=252):
    #annualised mean return minus variance penalty
    return r.mean() * periods - (gamma / 2) * r.var() * periods


def quarterly_sharpes(chunks):
    #create annualised sharpe ratio for each quarterly test window
    return np.array([sharpe_lo2002(c) for c in chunks])


def bootstrap_sharpe_diff(chunks_a, chunks_b, n_boot=5000, seed=42, block_length=4):
    #create circular block bootstrap of paired window Sharpes; null-centred two-sided test
    #Default block length is four quarterly windows, with sensitivity reported below.
    #This preserves local dependence within sampled blocks, not all dependence.
    if len(chunks_a) != len(chunks_b) or len(chunks_a) < 2:
        raise ValueError('Need at least two matching windows.')
    for a, b in zip(chunks_a, chunks_b):
        if not a.index.equals(b.index):
            raise ValueError('Paired return dates differ.')
    diffs = quarterly_sharpes(chunks_a) - quarterly_sharpes(chunks_b)
    if not np.isfinite(diffs).all():
        raise ValueError('Non-finite window Sharpe; inspect returns before inference.')
    n = len(diffs)
    length = int(block_length)
    if not 1 <= length <= n or n_boot < 100:
        raise ValueError('Invalid block length or bootstrap count.')
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, n, size=(n_boot, int(np.ceil(n / length))))
    indices = ((starts[..., None] + np.arange(length)) % n).reshape(n_boot, -1)[:, :n]
    obs = float(diffs.mean())
    null_means = (diffs - obs)[indices].mean(axis=1)
    q_lo, q_hi = np.quantile(null_means, [0.025, 0.975])
    p_value = (1 + np.count_nonzero(np.abs(null_means) >= abs(obs))) / (n_boot + 1)
    return obs, float(obs - q_hi), float(obs - q_lo), float(p_value)
print("Metrics and circular block bootstrap defined.")


## 5. Distance candidates

### 5.1 Correlation baseline

* Transform Pearson correlation to `sqrt((1 - correlation) / 2)`, then fit HRP using training data and apply the resulting target weights to each test day. All candidates reuse this fit/evaluate pattern.
* These are gross returns under constant daily target weights.


In [ ]:
# Correlation-HRP baseline. Refit each training window and evaluate the next block.
# Return chunks are kept for paired comparisons; the pooled series is used for metrics.

# Candidate 1: Correlation distance (baseline)
base_rets, base_chunks, base_turnover = [], [], []
prev_w = None
#tr: training window, tst: out-of-sample test window
for ts, te, vs, ve in windows:
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    #fit distance matrix on training data only
    dist = get_corr_distance(tr)
    w = hrp_weights_from_dist(tr, dist)
    # Apply the fitted target weights to each daily test return.
    # Apply the same target weights every day in the test period.
    r = (tst @ w[SYMBOLS]).rename('CorrHRP')
    #applying weights to test window's returns
    base_rets.append(r); base_chunks.append(r)
    if prev_w is not None:
        # Full L1 change between successive quarterly target vectors.
        base_turnover.append(np.abs(w - prev_w).sum())
    #weight change vs previous window's weights
    prev_w = w
base_series = pd.concat(base_rets)
print("=== Candidate 1: Correlation-distance HRP (baseline) ===")
print(f"Sharpe (Lo 2002): {sharpe_lo2002(base_series):.3f}")
print(f"Max Drawdown: {max_drawdown(base_series):.4f}")
print(f"Avg quarterly turnover: {np.mean(base_turnover):.4f}")

### 5.2 VARLiNGAM contemporaneous distance

* Fit VARLiNGAM to training returns and retain only its contemporaneous adjacency matrix `B0`.
* The existing configuration uses BIC with a maximum lag of one, the default DirectLiNGAM component and adaptive-Lasso pruning.

* Average `abs(B0)` with its transpose to discard direction and sign.
* Divide by the largest strength plus a small numerical offset, subtract from one, then set the diagonal to zero.
* Stronger fitted links give smaller dissimilarities.
* Lagged adjacency matrices do not enter this distance.

* These are model-estimated relationships under causal identification assumptions.
* Symmetrization does not establish causality or guarantee the triangle inequality.


In [ ]:
# VARLiNGAM distance uses contemporaneous effects only.
# Direction and coefficient sign are discarded before clustering.

# Candidate 2: VARLiNGAM causal distance
def get_varlingam_distance(train_returns):
    # Select lag order by BIC, allowing at most one lag.
    # Use default DirectLiNGAM and adaptive-Lasso pruning.
    model = VARLiNGAM(lags=1, criterion='bic', prune=True)
    model.fit(train_returns.values)

    # Extract contemporaneous effects (lag zero).
    B0 = model.adjacency_matrices_[0]
    # Symmetrize contemporaneous effects
    causal_sym = (np.abs(B0) + np.abs(B0.T)) / 2
    np.fill_diagonal(causal_sym, 0)
    # Normalize with stability epsilon
    # If all strengths are zero, every off-diagonal distance is one.
    # Clustering then depends on tie handling rather than differentiated causal strengths.
    dist = 1 - causal_sym / (causal_sym.max() + 1e-8)
    np.fill_diagonal(dist, 0)
    return pd.DataFrame(dist, index=train_returns.columns, columns=train_returns.columns)

vl_rets, vl_chunks, vl_turnover = [], [], []
prev_w_vl = None
#tr: training window, tst: out-of-sample test window
for ts, te, vs, ve in windows:
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    #fit varlingam distance on training data
    dist = get_varlingam_distance(tr)
    #fitting HRP weights on training data
    w = hrp_weights_from_dist(tr, dist)
    # Apply the same target weights every day in the test period.
    r = (tst @ w[SYMBOLS]).rename('VARLiNGAM-HRP')
    #applying weights to test window's returns
    vl_rets.append(r); vl_chunks.append(r)
    if prev_w_vl is not None:
        # Full L1 change between successive quarterly target vectors.
        # Initial allocation and within-quarter maintenance trades are excluded.
        vl_turnover.append(np.abs(w - prev_w_vl).sum())
    #weight change vs previous window's weights
    prev_w_vl = w
vl_series = pd.concat(vl_rets)
print("VARLiNGAM backtest complete.")

### 5.3 Granger-masked VAR distance

* Fit a joint VAR(1), then use separate bivariate F-tests to decide which off-diagonal coefficients to keep.
* Each test asks whether the second series helps predict the first beyond its own lag. This is predictive Granger causality
* Absolute masked coefficients are symmetrized and converted to dissimilarities using the same transformation as VARLiNGAM.

The pairwise 5% masking threshold is unadjusted across asset pairs.


In [ ]:
# Combine pairwise predictive tests with coefficients from a joint VAR(1).
# The per-edge p-value threshold is unadjusted across the directed pairs.

# Candidate 3: Granger causal distance
def get_granger_distance(train_returns, pvalue_threshold=0.05):
    cols = list(train_returns.columns)
    n = len(cols)
    #joint VAR(1) fit across all n assets
    var_result = VAR(train_returns).fit(1)
    # lag-1 coefficient matrix from the joint model
    B_var = var_result.coefs[0]
    #marking (i,j) pairs that are significant
    mask = np.eye(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            # The second column is tested as a predictor of the first (j -> i).
            # The pairwise regression does not condition on the other stocks in the joint VAR.
            bivariate = train_returns[[cols[i], cols[j]]].values
            # pairwise F-test for j -> i
            gc = grangercausalitytests(bivariate, maxlag=1)
            p_val = gc[1][0]['ssr_ftest'][1]
            if p_val < pvalue_threshold:
                #keeping edge only if pairwise test is significant
                mask[i, j] = True
    #apply pairwise-significance mask to joint VAR coefficient
    A_g = B_var * mask
    # symmetrize (same construction as Candidate 2)
    causal_sym = (np.abs(A_g) + np.abs(A_g.T)) / 2
    np.fill_diagonal(causal_sym, 0)
    dist = 1 - causal_sym / (causal_sym.max() + 1e-8)
    np.fill_diagonal(dist, 0)
    return pd.DataFrame(dist, index=cols, columns=cols)
gr_rets, gr_chunks, gr_turnover = [], [], []
prev_w_gr = None
#tr: training window, tst: out-of-sample test window
for ts, te, vs, ve in windows:
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    #granger distance fit on training data only
    dist = get_granger_distance(tr)
    #HRP weights fit on training data only
    w = hrp_weights_from_dist(tr, dist)
    # Apply the same target weights every day in the test period.
    r = (tst @ w[SYMBOLS]).rename('Granger-HRP')
    gr_rets.append(r); gr_chunks.append(r)
    if prev_w_gr is not None:
        #weight change vs previous window's weights
        gr_turnover.append(np.abs(w - prev_w_gr).sum())
    prev_w_gr = w
#stitch all 79 out-of-sample windows into one series
gr_series = pd.concat(gr_rets)
print("=== Candidate 3: Granger causal-distance HRP ===")
print(f"Sharpe (Lo 2002): {sharpe_lo2002(gr_series):.3f}")
print(f"Max Drawdown: {max_drawdown(gr_series):.4f}")
print(f"Avg quarterly turnover: {np.mean(gr_turnover):.4f}")

### 5.4 PCA distance

* Standardize each asset’s training returns, retain three principal component vectors, and represent each asset by its entries in those vectors. Compute Euclidean distances between these three-coordinate representations.


In [ ]:
# Fit PCA on standardized training returns and compare asset component coordinates.
# The default representation uses three component vectors.

# Candidate 4: PCA-based distance
def get_pca_distance(train_returns, n_components=3):
    X = train_returns.values
    # Standardize to prevent volatility bias per elaboration
    X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    pca = PCA(n_components=n_components)
    pca.fit(X_std)
    # each asset's loading vector on the top-3 components
    # Shape: number of assets × retained components. These are eigenvector
    # coordinates, without explained-variance scaling.
    loadings = pca.components_.T
    n = loadings.shape[0]
    D = np.zeros((n, n))
    # Euclidean distance in factor loading space
    for i in range(n):
        diff = loadings - loadings[i]
        D[i] = np.sqrt((diff**2).sum(axis=1))
    np.fill_diagonal(D, 0.0)
    return pd.DataFrame(D, index=train_returns.columns, columns=train_returns.columns)

pca_rets, pca_chunks, pca_turnover = [], [], []
prev_w_pca = None
for ts, te, vs, ve in windows:
    #tr: training window, tst: out-of-sample test window
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    #fit PCA distance on training data only
    dist = get_pca_distance(tr)
    #fit HRP weights on training data
    w = hrp_weights_from_dist(tr, dist)
    #apply weights to test window's returns
    # Apply the same target weights every day in the test period. This assumes
    # constant-mix holdings, rather than a buy-and-hold portfolio with drifting weights.
    r = (tst @ w[SYMBOLS]).rename('PCA-HRP')
    pca_rets.append(r); pca_chunks.append(r)
    if prev_w_pca is not None:
        #weight change vs previous window's weights
        # Full L1 change between successive quarterly target vectors.
        # Initial allocation and within-quarter maintenance trades are excluded.
        pca_turnover.append(np.abs(w - prev_w_pca).sum())
    prev_w_pca = w
#stitch all 79 out-of-sample windows into one series
pca_series = pd.concat(pca_rets)
print("PCA-HRP backtest complete.")

### 5.5 Factor-residual correlation distance

* Use the equal-weight return of these same ten stocks as a market proxy. Estimate each stock’s slope against that proxy and subtract its fitted slope contribution. An explicit intercept is unnecessary for the subsequent correlation because correlation centers the residual series.

* Transform residual correlation into distance.

* The next cell defines the additional Sortino and Calmar summaries used in the comparison.


In [ ]:
# Cluster using correlations after subtracting the equal-weight proxy contribution.
# Raw-return covariance and raw test returns are still used for allocation and evaluation.

# Candidate 5: Factor-model distance (residual correlation after removing an equal-weight market factor)
def get_factor_distance(train_returns):
    #equal-weight market proxy, one value per day
    market = train_returns.mean(axis=1)
    resid = pd.DataFrame(index=train_returns.index, columns=train_returns.columns, dtype=float)
    x = market.values
    for col in train_returns.columns:
        y = train_returns[col].values
        #single-factor market beta for this asset
        # The slope formula assumes the proxy has nonzero variance.
        # Each asset is also a constituent of this equal-weight proxy.
        beta = np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1)
        #idiosyncratic (residual) return, market component removed
        # Subtract beta times the proxy. Omitting the intercept leaves only a constant
        # shift relative to OLS residuals, which does not change residual correlation.
        resid[col] = y - beta * x
    #correlation of residuals,  not raw returns
    corr = resid.corr()
    # Apply the correlation-distance transform used by the baseline.
    return np.sqrt((1 - corr) / 2)
fac_rets, fac_chunks, fac_turnover = [], [], []
prev_w_fac = None
for ts, te, vs, ve in windows:
    #tr: training window, tst: out-of-sample test window
    tr, tst = returns.loc[ts:te], returns.loc[vs:ve]
    # factor-model distance fit on training data only
    dist = get_factor_distance(tr)
    # HRP weights fit on training data only
    w = hrp_weights_from_dist(tr, dist)
    # apply those weights to the test window's returns
    # Apply the same target weights every day in the test period. This assumes
    # constant-mix holdings, rather than a buy-and-hold portfolio with drifting weights.
    r = (tst @ w[SYMBOLS]).rename('Factor-HRP')
    fac_rets.append(r); fac_chunks.append(r)
    if prev_w_fac is not None:
        # weight change vs. previous window's weights
        # Initial allocation and within-quarter maintenance trades are excluded.
        fac_turnover.append(np.abs(w - prev_w_fac).sum())
    prev_w_fac = w
# stitch all 79 out-of-sample windows into 1 series
fac_series = pd.concat(fac_rets)
print("=== Candidate 5: Factor-model distance HRP ===")
print(f"Sharpe (Lo 2002): {sharpe_lo2002(fac_series):.3f}")
print(f"Max Drawdown: {max_drawdown(fac_series):.4f}")
print(f"Avg quarterly turnover: {np.mean(fac_turnover):.4f}")

In [ ]:
# Additional ratios and a metric-agnostic paired-window bootstrap.
# Calmar uses annualized arithmetic mean return here; Sortino uses a daily target.

# Additional metrics: Sortino, Calmar, and a generalized bootstrap significance test
def sortino_ratio(r, periods=252, target=0.0):
    diff = r - target
    downside_sq = np.minimum(diff, 0.0) ** 2
    # Full-sample downside deviation per Sortino and Price (1994): averaged over all periods, not just downside ones
    # Average squared shortfalls over ALL days, including zeros for non-shortfalls.
    # A nonempty series with no downside has denominator zero; this case is not guarded.
    dd = np.sqrt(downside_sq.mean()) * np.sqrt(periods) if len(r) > 0 else 1e-12
    ann_ret = r.mean() * periods - target * periods
    return ann_ret / dd

def calmar_ratio(r, periods=252):
    # Use annualized arithmetic mean rather than a compounded annual return.
    # Corrected 2026-09-09: the denominator now uses max_drawdown's corrected wealth path (initial wealth 1.0 included).
    ann_ret = r.mean() * periods
    mdd = abs(max_drawdown(r))
    # Prevent division by zero
    return ann_ret / mdd if mdd > 1e-12 else np.nan

def bootstrap_metric_diff(chunks_a, chunks_b, metric_fn, n_boot=5000, seed=42, block_length=4):
    """Circular block bootstrap of paired window metric scores; null-centred two-sided test.

    Default block length is four quarterly windows, with sensitivity reported below.
    This preserves local dependence within sampled blocks, not all dependence.
    Short regime samples remain exploratory. CI is a basic bootstrap interval.
    """
    if len(chunks_a) != len(chunks_b) or len(chunks_a) < 2:
        raise ValueError('Need at least two matching windows.')
    for a, b in zip(chunks_a, chunks_b):
        if not a.index.equals(b.index):
            raise ValueError('Paired return dates differ.')
    diffs = np.array([metric_fn(c) for c in chunks_a]) - np.array([metric_fn(c) for c in chunks_b])
    if not np.isfinite(diffs).all():
        raise ValueError('Non-finite window metric; inspect returns before inference.')
    n = len(diffs)
    length = int(block_length)
    if not 1 <= length <= n or n_boot < 100:
        raise ValueError('Invalid block length or bootstrap count.')
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, n, size=(n_boot, int(np.ceil(n / length))))
    indices = ((starts[..., None] + np.arange(length)) % n).reshape(n_boot, -1)[:, :n]
    obs = float(diffs.mean())
    null_means = (diffs - obs)[indices].mean(axis=1)
    q_lo, q_hi = np.quantile(null_means, [0.025, 0.975])
    p_value = (1 + np.count_nonzero(np.abs(null_means) >= abs(obs))) / (n_boot + 1)
    return obs, float(obs - q_hi), float(obs - q_lo), float(p_value)


## 6. Cross-candidate comparison

* The first table reports pooled daily-return metrics.
* The next tables report mean paired window-level differences against correlation, 95% basic intervals and unadjusted null-centred p-values from four-window circular blocks (5,000 draws; seed 42).

* Positive differences favour the candidate for Sharpe, CEQ and negative-valued MaxDD.
* A negative MaxDD difference means worse average window drawdown, even if the pooled drawdown ranking differs.

* Holm correction covers the 12 comparisons here (four candidates × three metrics).
* The later 20-test regime family is separate. Neither correction covers the complete historical model search.


In [ ]:
# Each candidate stores (pooled return series, ordered test chunks, target turnover).
# This table summarizes pooled performance; it does not select a winner automatically.

# Full summary across all metrics, significance testing, and champion determination
candidates = {
    'Correlation (baseline)': (base_series, base_chunks, base_turnover),
    'VARLiNGAM': (vl_series, vl_chunks, vl_turnover),
    'Granger': (gr_series, gr_chunks, gr_turnover),
    'PCA': (pca_series, pca_chunks, pca_turnover),
    'Factor-model': (fac_series, fac_chunks, fac_turnover),
}
rows = []
for name, (series, chunks, turnover) in candidates.items():
    rows.append({
        'Sharpe': sharpe_lo2002(series),
        'Sortino': sortino_ratio(series),
        'Calmar': calmar_ratio(series),
        'MaxDD': max_drawdown(series),
        'CEQ (g=1)': ceq(series, 1),
        'CEQ (g=3)': ceq(series, 3),
        'Avg Turnover': np.mean(turnover),
    })
summary = pd.DataFrame(rows, index=list(candidates.keys()))
print("=== Phase 1 Fixed-Distance Grid: Full Summary ===")
print(summary.round(4).to_string())

In [ ]:
# Four-window circular blocks; null-centred test; unadjusted p-values.
#reconstructed significance table vs. correlation baseline
sig_names = ['VARLiNGAM', 'Granger', 'PCA', 'Factor-model']
baseline_chunks = candidates['Correlation (baseline)'][1]
results = {n: (bootstrap_metric_diff(candidates[n][1], baseline_chunks, sharpe_lo2002), bootstrap_metric_diff(candidates[n][1], baseline_chunks, max_drawdown), bootstrap_metric_diff(candidates[n][1], baseline_chunks, lambda r: ceq(r, 1))) for n in sig_names}
sig_table = pd.DataFrame({n: {'Sharpe': results[n][0][0], 'Sharpe p': results[n][0][3], 'MaxDD': results[n][1][0], 'MaxDD p': results[n][1][3], 'CEQ': results[n][2][0], 'CEQ p': results[n][2][3]} for n in sig_names}).T
print("=== Reconstructed significance table vs. Correlation baseline (block bootstrap, seed=42) ===")
print(sig_table.round(4).to_string())

In [ ]:
# Reuse the bootstrap results above to format 95% basic intervals.
# The strings round for display; they do not alter the stored numeric results.

#confidence interval tables
ci_rows = {n: {'Sharpe CI': f'[{results[n][0][1]:+.4f}, {results[n][0][2]:+.4f}]', 'MaxDD CI': f'[{results[n][1][1]:+.4f}, {results[n][1][2]:+.4f}]', 'CEQ CI': f'[{results[n][2][1]:+.4f}, {results[n][2][2]:+.4f}]'} for n in sig_names}
ci_table = pd.DataFrame(ci_rows).T
print(ci_table.to_string())

In [ ]:
#holm-bonferroni corrections

from statsmodels.stats.multitest import multipletests

pval_labels, pvals = [], []
for name in sig_table.index:
    for metric in ['Sharpe', 'MaxDD', 'CEQ']:
        pval_labels.append(f'{name} {metric}')
        pvals.append(sig_table.loc[name, f'{metric} p'])

# Holm controls this family only when its supplied p-values are valid.
reject, pvals_holm, _, _ = multipletests(pvals, alpha=0.05, method='holm')
holm_df = pd.DataFrame({'p_raw': pvals, 'p_holm': pvals_holm, 'survives_0.05': reject}, index=pval_labels)
print("=== Holm-Bonferroni correction across all 12 fixed-distance-grid significance tests ===")
print(holm_df.round(4).to_string())

## 7. Robustness check: PCA component sensitivity

* Repeat PCA-HRP with 1, 2, 3, 5 and 7 components using the same windows. Each row combines pooled Sharpe with a mean window Sharpe difference and bootstrap interval versus correlation.

* This is an exploratory parameter sweep.
* Selecting the component count from these test results reuses the evaluation sample; these five comparisons are not included in the earlier 12-test Holm family.


In [ ]:
# Sensitivity sweep over the retained PCA dimension. All settings share test dates.
# Pooled Sharpe and mean paired window differences are distinct summaries.

# Check how sensitive the results are to the number of PCA components.
pca_k_results = []

for k in [1, 2, 3, 5, 7]:
    k_chunks = []

    # use the same training and test windows as the main comparison.
    for ts, te, vs, ve in windows:
        tr = returns.loc[ts:te]
        tst = returns.loc[vs:ve]
        # estimate distances and portfolio weights using training data only.
        dist = get_pca_distance(tr, n_components=k)
        w = hrp_weights_from_dist(tr, dist)

        # Apply the target weights to each day's out-of-sample returns.
        # This assumes constant weights within the test window.
        r = (tst @ w[SYMBOLS]).rename(f'PCA{k}-HRP')
        k_chunks.append(r)

    # Combine the test windows to calculate the full-sample Sharpe ratio.
    k_series = pd.concat(k_chunks)
    # comparison
    delta, ci_lo, ci_hi, p_value = bootstrap_metric_diff(
        k_chunks, base_chunks, sharpe_lo2002
    )

    pca_k_results.append({
        'k': k,
        'Sharpe': sharpe_lo2002(k_series),
        'Delta': delta,
        'CI_lo': ci_lo,
        'CI_hi': ci_hi,
        'p': p_value,
    })

pca_k_df = pd.DataFrame(pca_k_results).set_index('k')
print('=== PCA Component Sensitivity (k=1,2,3,5,7) ===')
print(pca_k_df.round(4).to_string())

## 8. Robustness check: Chronological half-split

* Split the ordered chunk lists at `len(windows) // 2`, then calculate pooled Sharpe and Calmar separately in each half.
* With 79 windows, this gives 39 and 40 windows.


In [ ]:
# Robustness B: Chronological Half-Split Stability
# Checking if performance is driven by a specific time period
n_half = len(windows) // 2
half_rows = []
for name, (series, chunks, turnover) in candidates.items():
    first_series = pd.concat(chunks[:n_half])
    second_series = pd.concat(chunks[n_half:])
    half_rows.append({
        'Sharpe (1st)': sharpe_lo2002(first_series),
        'Sharpe (2nd)': sharpe_lo2002(second_series),
        'Calmar (1st)': calmar_ratio(first_series),
        'Calmar (2nd)': calmar_ratio(second_series),
    })
half_df = pd.DataFrame(half_rows, index=list(candidates.keys()))
print("=== Chronological Stability (2006-2016 vs 2016-2026) ===")
display(half_df.round(4))

## 9. Robustness check: Market-regime comparisons

* Assign each complete test window by its start date: Pre-GFC before 2008; GFC from 2008 through 2009; Post-GFC-to-COVID from 2010 through 2019; COVID+ from 2020.
* Bounds include the lower date and exclude the upper date. Windows crossing a boundary retain the starting label.

* These are calendar buckets, not estimated stress states.
* The first diagnostic groups Pre-GFC and GFC inside the first 39 windows; its “crisis” and “calm” variable names are shorthand and do not classify observed volatility.
* The regime wrapper displays pooled metrics beside mean window differences.
* Five comparisons across four regimes form the separate 20-test Holm family. This does not correct the entire notebook as a single family.
* Within-regime inference uses four-window circular blocks. Short regime samples remain exploratory; circular wrapping joins the end and beginning of each regime.


In [ ]:
# Define nonoverlapping calendar buckets with inclusive lower and exclusive upper bounds.
# The label is determined solely by each test window’s start date.

#defining 4 cal regimes
regime_bounds = [('Pre-GFC', pd.Timestamp('1900-01-01'), pd.Timestamp('2008-01-01')), ('GFC', pd.Timestamp('2008-01-01'), pd.Timestamp('2010-01-01')), ('Post-GFC-to-COVID', pd.Timestamp('2010-01-01'), pd.Timestamp('2020-01-01')), ('COVID+', pd.Timestamp('2020-01-01'), pd.Timestamp('2100-01-01'))]

#return regime containing date d
regime_of = lambda d: next(name for name, lo, hi in regime_bounds if lo <= d < hi)

#assign regime to each window
regime_labels = [regime_of(w[2]) for w in windows]

print('Regime window counts:', {name: regime_labels.count(name) for name, _, _ in regime_bounds})

In [ ]:
# Diagnostic for the first 39 windows only; 39 is a fixed reference-sample choice.
# This slicing assumes Pre-GFC/GFC windows form the leading contiguous block.
# The label “crisis” includes Pre-GFC, and “calm” is a calendar shorthand.
# AnnRet is annualized arithmetic mean return.

#count pre-GFC and GFC windows within the first 39 eval windows
n_crisis = regime_labels[:39].count('Pre-GFC') + regime_labels[:39].count('GFC')

#extract return-series chunks for each strategy
corr_chunks, fac_chunks = candidates['Correlation (baseline)'][1], candidates['Factor-model'][1]

#combinining returns into 3 samples (crisis, calm and full first half)
corr_crisis, corr_calm, corr_full = pd.concat(corr_chunks[:n_crisis]), pd.concat(corr_chunks[n_crisis:39]), pd.concat(corr_chunks[:39])
fac_crisis, fac_calm, fac_full = pd.concat(fac_chunks[:n_crisis]), pd.concat(fac_chunks[n_crisis:39]), pd.concat(fac_chunks[:39])

#compare performance across subsamples
print(f'First-half composition: Pre-GFC+GFC={n_crisis} windows, remaining Post-GFC-to-COVID={39-n_crisis} windows')
print(f'Correlation  | crisis Calmar={calmar_ratio(corr_crisis):.4f} MaxDD={max_drawdown(corr_crisis):.4f} AnnRet={corr_crisis.mean()*252:.4f} | calm Calmar={calmar_ratio(corr_calm):.4f} MaxDD={max_drawdown(corr_calm):.4f} AnnRet={corr_calm.mean()*252:.4f} | fullfirsthalf Calmar={calmar_ratio(corr_full):.4f} MaxDD={max_drawdown(corr_full):.4f}')
print(f'Factor-model | crisis Calmar={calmar_ratio(fac_crisis):.4f} MaxDD={max_drawdown(fac_crisis):.4f} AnnRet={fac_crisis.mean()*252:.4f} | calm Calmar={calmar_ratio(fac_calm):.4f} MaxDD={max_drawdown(fac_calm):.4f} AnnRet={fac_calm.mean()*252:.4f} | fullfirsthalf Calmar={calmar_ratio(fac_full):.4f} MaxDD={max_drawdown(fac_full):.4f}')

In [ ]:
# Compare two candidate chunk lists within each calendar regime.
# Pooled metric columns and the mean window delta summarize different quantities.
# All regimes must be nonempty; this wrapper has no empty-subset guard.

def regime_table(name_a, name_b, metric_fn=None):

    # use Sharpe unless another metric is supplied.
    if metric_fn is None:
        metric_fn = sharpe_lo2002

    rows = []

    for regime_name, _, _ in regime_bounds:
        # regime labels are assigned using each test window's start date.
        window_indices = [
            i for i, label in enumerate(regime_labels)
            if label == regime_name
        ]

        # select matching windows so both candidates cover the same dates.
        chunks_a = [candidates[name_a][1][i] for i in window_indices]
        chunks_b = [candidates[name_b][1][i] for i in window_indices]

        # bootstrap the paired differences in per-window metric scores.
        delta, ci_lo, ci_hi, p_value = bootstrap_metric_diff(
            chunks_a, chunks_b, metric_fn, n_boot=5000, seed=42
        )

        #p values are unadjusted
        # Report pooled candidate metrics alongside window-average differences.
        # A duplicated “(baseline)” in the column label is cosmetic: name_b already includes it.
        rows.append({
            'Regime': regime_name,
            'N': len(window_indices),
            name_b + ' (baseline)': metric_fn(pd.concat(chunks_b)),
            name_a: metric_fn(pd.concat(chunks_a)),
            'DeltaVsBaseline': delta,
            'CI_lo': ci_lo,
            'CI_hi': ci_hi,
            'p_value': p_value,
        })

    return pd.DataFrame(rows).set_index('Regime')

In [ ]:
# Create four Sharpe comparisons and one Factor-model Calmar comparison per regime.
# The fifth call overrides the default metric; all calls keep the same paired windows.

#compare each candidate against correlation baseline within each regime using a specific metric
t_varlingam = regime_table('VARLiNGAM', 'Correlation (baseline)')
t_granger = regime_table('Granger', 'Correlation (baseline)')
t_pca = regime_table('PCA', 'Correlation (baseline)')
t_factor = regime_table('Factor-model', 'Correlation (baseline)')
t_factor_calmar = regime_table('Factor-model', 'Correlation (baseline)', metric_fn=calmar_ratio)

#printing each regime comparison
print('=== VARLiNGAM vs Correlation: Sharpe regime split ==='); print(t_varlingam.round(4).to_string())
print(); print('=== Granger vs Correlation: Sharpe regime split ==='); print(t_granger.round(4).to_string())
print(); print('=== PCA vs Correlation: Sharpe regime split ==='); print(t_pca.round(4).to_string())
print(); print('=== Factor-model vs Correlation: Sharpe regime split ==='); print(t_factor.round(4).to_string())
print(); print('=== Factor-model vs Correlation: Calmar regime split ==='); print(t_factor_calmar.round(4).to_string())

In [ ]:
#holm-bonferroni correction

#collecting the comparison tables
regime_tables = {'VARLiNGAM Sharpe': t_varlingam, 'Granger Sharpe': t_granger, 'PCA Sharpe': t_pca, 'Factor-model Sharpe': t_factor, 'Factor-model Calmar': t_factor_calmar}

#gathering each test's label and unadjusted p val
pval_labels2, pvals2 = [], []
for label, tbl in regime_tables.items():
    for regime in tbl.index:
        pval_labels2.append(f'{label} [{regime}]')
        pvals2.append(tbl.loc[regime, 'p_value'])

# Keep full-precision p-values for correction; round only the displayed table.
# The earlier overall Holm correction and this correction are separate families.
reject2, pvals_holm2, _, _ = multipletests(pvals2, alpha=0.05, method='holm')
holm_regime_df = pd.DataFrame({'p_raw': pvals2, 'p_holm': pvals_holm2, 'survives_0.05': reject2}, index=pval_labels2)
print("=== Holm-Bonferroni correction across all 20 Phase 1 regime-grid significance tests ===")
print(holm_regime_df.round(4).to_string())

## 10. Summary chart

Plot pooled Sharpe, Sortino and the implemented arithmetic-return Calmar for each candidate.


In [ ]:
# Plot the pooled summary ratios, grouped by candidate.
# The bars are point estimates with different risk denominators and no intervals.

import numpy as np
import matplotlib.pyplot as plt

metrics = ['Sharpe', 'Sortino', 'Calmar']
labels = list(summary.index)
data = summary.loc[labels, metrics].values
x = np.arange(len(labels))
width = 0.25
colors = ['#4C72B0', '#55A868', '#C44E52']
offsets = [-width, 0, width]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x + offsets[0], data[:, 0], width, label=metrics[0], color=colors[0])
ax.bar(x + offsets[1], data[:, 1], width, label=metrics[1], color=colors[1])
ax.bar(x + offsets[2], data[:, 2], width, label=metrics[2], color=colors[2])
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=0)
ax.set_ylabel('Ratio')
ax.set_title('Fixed-distance grid: Sharpe, Sortino, and Calmar by candidate (79 windows)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_phase1_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print(summary[metrics].round(4))
